# NYT Connections Dataset — Comprehensive Exploration

Deep analysis of 554 NYT Connections puzzles (June 2023 – December 2024).

**Sections:**
1. Dataset Overview
2. Word Analysis
3. Category Analysis
4. Difficulty Analysis
5. Embedding Analysis (MPNET cosine similarity by color)
6. Cross-Puzzle Patterns

In [ ]:
import os, sys, json, re, warnings
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (11, 5),
                      'axes.titlesize': 13, 'axes.labelsize': 11})

ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
sys.path.insert(0, str(ROOT))

DATA_PATH = ROOT / 'data' / 'nyt_puzzles' / 'ConnectionsFinalDataset (1).json'
with open(DATA_PATH) as f:
    raw = json.load(f)

print(f'Loaded {len(raw)} puzzles from {DATA_PATH.name}')

# Accumulator for stats we'll save at the end
STATS = {}

---
## 1. Dataset Overview

In [ ]:
# Parse dates and build a tidy dataframe
rows = []
for p in raw:
    dt = datetime.strptime(p['date'], '%Y/%m/%d')
    rows.append({
        'date': dt,
        'contest': p.get('contest', ''),
        'difficulty': p.get('difficulty'),
        'num_words': len(p['words']),
        'num_answers': len(p['answers']),
    })
df_puzzles = pd.DataFrame(rows).sort_values('date').reset_index(drop=True)

date_min, date_max = df_puzzles['date'].min(), df_puzzles['date'].max()
n_puzzles = len(df_puzzles)
n_with_diff = df_puzzles['difficulty'].notna().sum()

STATS['dataset'] = {
    'total_puzzles': n_puzzles,
    'date_range': f"{date_min:%Y-%m-%d} to {date_max:%Y-%m-%d}",
    'puzzles_with_difficulty': int(n_with_diff),
}

print(f'Total puzzles:           {n_puzzles}')
print(f'Date range:              {date_min:%B %d, %Y} — {date_max:%B %d, %Y}')
print(f'Span:                    {(date_max - date_min).days} days')
print(f'Words per puzzle:        always {df_puzzles["num_words"].unique()}')
print(f'Answers per puzzle:      always {df_puzzles["num_answers"].unique()}')
print(f'Puzzles with difficulty:  {n_with_diff} / {n_puzzles} ({n_with_diff/n_puzzles:.0%})')

In [ ]:
# Puzzles per month
df_puzzles['month'] = df_puzzles['date'].dt.to_period('M')
monthly = df_puzzles.groupby('month').size()

fig, ax = plt.subplots(figsize=(13, 4.5))
monthly.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white', width=0.8)
ax.set_ylabel('Puzzles')
ax.set_xlabel('')
ax.set_title('Puzzles Published per Month')
# Rotate x labels
ax.set_xticklabels([str(p) for p in monthly.index], rotation=45, ha='right', fontsize=9)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()

STATS['dataset']['puzzles_per_month_avg'] = round(float(monthly.mean()), 1)

---
## 2. Word Analysis

In [ ]:
# Collect all words
all_words = []
word_puzzle_map = defaultdict(list)  # word -> list of puzzle indices

for i, p in enumerate(raw):
    for w in p['words']:
        wu = w.upper()
        all_words.append(wu)
        word_puzzle_map[wu].append(i)

word_counts = Counter(all_words)
unique_words = sorted(set(all_words))

STATS['words'] = {
    'total_appearances': len(all_words),
    'unique_words': len(unique_words),
    'avg_per_puzzle': round(len(all_words) / n_puzzles, 1),
}

print(f'Total word appearances: {len(all_words):,}')
print(f'Unique words:          {len(unique_words):,}')
print(f'Words per puzzle:      {len(all_words) / n_puzzles:.0f} (always 16)')
print(f'\nWords appearing only once: {sum(1 for c in word_counts.values() if c == 1):,}')
print(f'Words appearing 2+ times: {sum(1 for c in word_counts.values() if c >= 2):,}')
print(f'Words appearing 5+ times: {sum(1 for c in word_counts.values() if c >= 5):,}')

In [ ]:
# Top 30 most frequent words
top30 = word_counts.most_common(30)

fig, ax = plt.subplots(figsize=(13, 5))
words_list, counts_list = zip(*top30)
y_pos = np.arange(len(words_list))
ax.barh(y_pos, counts_list, color='steelblue', edgecolor='white')
ax.set_yticks(y_pos)
ax.set_yticklabels(words_list, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Appearances across 554 puzzles')
ax.set_title('Top 30 Most Frequent Words')
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
# Add count labels
for i, (w, c) in enumerate(top30):
    ax.text(c + 0.3, i, str(c), va='center', fontsize=8)
plt.tight_layout()
plt.show()

STATS['words']['top_10'] = [{'word': w, 'count': c} for w, c in word_counts.most_common(10)]

In [ ]:
# Word length distribution
word_lengths = [len(w) for w in unique_words]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Character length
axes[0].hist(word_lengths, bins=range(1, max(word_lengths)+2), color='steelblue',
             edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Word length (characters)')
axes[0].set_ylabel('Number of unique words')
axes[0].set_title('Word Length Distribution (characters)')
axes[0].axvline(np.median(word_lengths), color='coral', ls='--', label=f'median={np.median(word_lengths):.0f}')
axes[0].legend()

# Multi-word vs single-word
multi_word = [w for w in unique_words if ' ' in w]
single_word = [w for w in unique_words if ' ' not in w]
sizes = [len(single_word), len(multi_word)]
axes[1].pie(sizes, labels=[f'Single word\n({sizes[0]:,})', f'Multi-word\n({sizes[1]:,})'],
            autopct='%1.1f%%', colors=['steelblue', 'coral'], startangle=90)
axes[1].set_title('Single vs Multi-Word Entries')

plt.tight_layout()
plt.show()

STATS['words']['median_length'] = int(np.median(word_lengths))
STATS['words']['multi_word_entries'] = len(multi_word)

In [ ]:
# Words appearing in the most different puzzles
repeat_words = [(w, len(puzzles)) for w, puzzles in word_puzzle_map.items() if len(puzzles) >= 3]
repeat_words.sort(key=lambda x: -x[1])

print(f'Words appearing in 3+ puzzles: {len(repeat_words)}')
print(f'\nTop 20 most-reused words:')
for w, c in repeat_words[:20]:
    print(f'  {w:20s}  {c} puzzles')

STATS['words']['reused_3plus'] = len(repeat_words)

---
## 3. Category Analysis

In [ ]:
# Collect all categories
all_categories = []
cat_to_puzzles = defaultdict(list)

for i, p in enumerate(raw):
    for ans in p['answers']:
        cat = ans['answerDescription'].upper().strip()
        all_categories.append(cat)
        cat_to_puzzles[cat].append(i)

cat_counts = Counter(all_categories)
unique_cats = len(set(all_categories))

STATS['categories'] = {
    'total_appearances': len(all_categories),
    'unique_categories': unique_cats,
    'categories_per_puzzle': 4,
}

print(f'Total category slots:  {len(all_categories)} (554 × 4)')
print(f'Unique category names: {unique_cats}')
print(f'Reuse rate:            {1 - unique_cats/len(all_categories):.1%} of slots use a repeated name')
print(f'\nCategories used more than once:')
repeated_cats = [(c, n) for c, n in cat_counts.most_common() if n > 1]
print(f'  {len(repeated_cats)} categories appear 2+ times')
for c, n in repeated_cats[:15]:
    print(f'  {c:40s}  {n}×')

In [ ]:
# Category name length distribution
cat_char_lens = [len(c) for c in set(all_categories)]
cat_word_lens = [len(c.split()) for c in set(all_categories)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].hist(cat_word_lens, bins=range(1, max(cat_word_lens)+2),
             color='coral', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Words in category name')
axes[0].set_ylabel('Count')
axes[0].set_title('Category Name Length (words)')
axes[0].axvline(np.median(cat_word_lens), color='steelblue', ls='--',
                label=f'median={np.median(cat_word_lens):.0f}')
axes[0].legend()

axes[1].hist(cat_char_lens, bins=30, color='coral', edgecolor='white', alpha=0.85)
axes[1].set_xlabel('Characters in category name')
axes[1].set_ylabel('Count')
axes[1].set_title('Category Name Length (characters)')
axes[1].axvline(np.median(cat_char_lens), color='steelblue', ls='--',
                label=f'median={np.median(cat_char_lens):.0f}')
axes[1].legend()

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

STATS['categories']['median_word_length'] = int(np.median(cat_word_lens))
STATS['categories']['median_char_length'] = int(np.median(cat_char_lens))

In [ ]:
# Categorize categories by type using pattern matching
def classify_category(cat: str) -> str:
    """Heuristic classification of category type."""
    c = cat.upper()
    # Fill-in-the-blank patterns: ___ or ___
    if '___' in c or '_ _' in c or re.search(r'\b\w+\s*_', c) or c.endswith('...'):
        return 'Fill-in-the-blank'
    if re.search(r'^_', c) or re.search(r'_$', c):
        return 'Fill-in-the-blank'
    # Wordplay: contains parenthetical, quotes, or explicit wordplay markers
    if '(' in c or '"' in c or "'" in c:
        return 'Wordplay / Hidden pattern'
    # Synonyms: "WAYS TO SAY", "WORDS MEANING", "ANOTHER WORD FOR"
    if any(p in c for p in ['WAYS TO', 'WORDS MEANING', 'SYNONYM', 'ANOTHER WORD',
                             'SLANG FOR', 'MEANS']):
        return 'Synonyms / Slang'
    # Pop culture: names, titles, brands
    pop_kw = ['MOVIE', 'FILM', 'SONG', 'ALBUM', 'TV', 'SHOW', 'BOOK', 'NOVEL',
              'BAND', 'ARTIST', 'ACTOR', 'CHARACTER', 'DISNEY', 'MARVEL', 'STAR WARS',
              'TAYLOR', 'BEYONCE', 'BEATLES', 'NBA', 'NFL', 'MLB', 'GAME',
              'VIDEO GAME', 'SERIES', 'MUSICAL', 'ANIME', 'BRAND', 'COMPANY']
    if any(kw in c for kw in pop_kw):
        return 'Pop culture'
    # Common property: "THINGS THAT", "TYPES OF", "KINDS OF", starts with adjective
    if any(p in c for p in ['THINGS', 'TYPES OF', 'KINDS OF', 'FORMS OF',
                             'PARTS OF', 'PIECES OF', 'SOURCES OF',
                             'ASSOCIATED WITH', 'FOUND IN', 'USED FOR',
                             'MADE OF', 'WITH A', 'THAT HAVE', 'THAT ARE',
                             'THAT CAN', 'ENDING IN', 'STARTING WITH',
                             'BEGINNING WITH', 'CONTAIN']):
        return 'Common property'
    # If it starts with a verb: common property / action grouping
    first_word = c.split()[0] if c.split() else ''
    if first_word in ['GO', 'GET', 'MAKE', 'TAKE', 'GIVE', 'PUT', 'SET',
                       'RUN', 'CUT', 'HIT', 'BREAK', 'TURN', 'COME', 'KEEP',
                       'PASS', 'PLAY', 'MOVE', 'OPEN', 'CLOSE', 'HOLD']:
        return 'Common property'
    # Catch remaining short categories as likely synonyms
    if len(c.split()) <= 2 and len(c) <= 15:
        return 'Synonyms / Slang'
    return 'Common property'


cat_types = [classify_category(c) for c in all_categories]
type_counts = Counter(cat_types)

print('Category type distribution:')
for t, n in type_counts.most_common():
    print(f'  {t:30s} {n:>4}  ({n/len(all_categories):.1%})')

STATS['categories']['type_distribution'] = dict(type_counts)

In [ ]:
# Plot category type distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
labels = [t for t, _ in type_counts.most_common()]
sizes = [c for _, c in type_counts.most_common()]
colors_pie = sns.color_palette('Set2', len(labels))
wedges, texts, autotexts = axes[0].pie(
    sizes, labels=labels, autopct='%1.1f%%', colors=colors_pie,
    startangle=90, pctdistance=0.75, textprops={'fontsize': 9}
)
axes[0].set_title('Category Types')

# Bar chart
axes[1].barh(labels[::-1], sizes[::-1], color=colors_pie[::-1], edgecolor='white')
axes[1].set_xlabel('Count (across 2,216 total category slots)')
axes[1].set_title('Category Type Counts')
for spine in ['top', 'right']:
    axes[1].spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

# Show examples per type
type_examples = defaultdict(list)
for cat, ctype in zip(all_categories, cat_types):
    if len(type_examples[ctype]) < 5:
        type_examples[ctype].append(cat)

print('\nExamples per category type:')
for ctype in type_counts:
    print(f'\n  {ctype}:')
    for ex in type_examples[ctype]:
        print(f'    • {ex}')

---
## 4. Difficulty Analysis

In [ ]:
# Difficulty distribution
df_diff = df_puzzles.dropna(subset=['difficulty']).copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df_diff['difficulty'], bins=20, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(df_diff['difficulty'].median(), color='coral', ls='--',
                label=f'median={df_diff["difficulty"].median():.2f}')
axes[0].axvline(df_diff['difficulty'].mean(), color='green', ls='--',
                label=f'mean={df_diff["difficulty"].mean():.2f}')
axes[0].set_xlabel('Difficulty Score')
axes[0].set_ylabel('Number of Puzzles')
axes[0].set_title(f'Difficulty Distribution ({len(df_diff)} rated puzzles)')
axes[0].legend()

# Box plot
axes[1].boxplot(df_diff['difficulty'], vert=True)
axes[1].set_ylabel('Difficulty Score')
axes[1].set_title('Difficulty Box Plot')
axes[1].set_xticklabels(['All Puzzles'])

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

STATS['difficulty'] = {
    'n_rated': len(df_diff),
    'mean': round(float(df_diff['difficulty'].mean()), 3),
    'median': round(float(df_diff['difficulty'].median()), 3),
    'std': round(float(df_diff['difficulty'].std()), 3),
    'min': round(float(df_diff['difficulty'].min()), 2),
    'max': round(float(df_diff['difficulty'].max()), 2),
}

print(f'Difficulty stats (n={len(df_diff)}):')
print(f'  Mean:   {df_diff["difficulty"].mean():.2f}')
print(f'  Median: {df_diff["difficulty"].median():.2f}')
print(f'  Std:    {df_diff["difficulty"].std():.2f}')
print(f'  Range:  {df_diff["difficulty"].min():.1f} – {df_diff["difficulty"].max():.1f}')

In [ ]:
# Easiest and hardest puzzles
easiest = df_diff.nsmallest(5, 'difficulty')[['date', 'difficulty']]
hardest = df_diff.nlargest(5, 'difficulty')[['date', 'difficulty']]

print('Top 5 EASIEST puzzles:')
for _, row in easiest.iterrows():
    print(f'  {row["date"]:%Y-%m-%d}  difficulty={row["difficulty"]:.2f}')

print('\nTop 5 HARDEST puzzles:')
for _, row in hardest.iterrows():
    print(f'  {row["date"]:%Y-%m-%d}  difficulty={row["difficulty"]:.2f}')

In [ ]:
# Difficulty trend over time
fig, ax = plt.subplots(figsize=(13, 5))

ax.scatter(df_diff['date'], df_diff['difficulty'], alpha=0.3, s=15, color='steelblue')

# Rolling average (30-puzzle window)
df_sorted = df_diff.sort_values('date')
rolling = df_sorted['difficulty'].rolling(window=30, center=True).mean()
ax.plot(df_sorted['date'], rolling, color='coral', linewidth=2.5, label='30-puzzle rolling mean')

ax.set_xlabel('Date')
ax.set_ylabel('Difficulty Score')
ax.set_title('Puzzle Difficulty Over Time — Are Newer Puzzles Harder?')
ax.legend()
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30, ha='right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

# Correlation
df_sorted['ordinal'] = range(len(df_sorted))
corr = df_sorted[['ordinal', 'difficulty']].corr().iloc[0, 1]
print(f'Correlation (puzzle order vs difficulty): r = {corr:.3f}')
if abs(corr) < 0.1:
    print('  → No significant trend: difficulty is essentially flat over time.')
elif corr > 0:
    print('  → Slight upward trend: newer puzzles are somewhat harder.')
else:
    print('  → Slight downward trend: newer puzzles are somewhat easier.')

STATS['difficulty']['time_correlation'] = round(float(corr), 4)

---
## 5. Embedding Analysis

Compute MPNET cosine similarity for every group and verify the color thresholds from the paper:

| Color | Expected Mean Similarity |
|-------|------------------------|
| Yellow | ~0.40 |
| Green | ~0.35 |
| Blue | ~0.29 |
| Purple | ~0.27 |

Since the dataset doesn't label colors per group, we assign them by ranking groups within each puzzle from highest similarity (yellow) to lowest (purple).

In [ ]:
# Load MPNET and compute similarities
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print('Loading MPNET model...')
model = SentenceTransformer('all-mpnet-base-v2')
print('Model loaded.')


def avg_pairwise_sim(words, model):
    """Average pairwise cosine similarity for a list of words."""
    if len(words) < 2:
        return 0.0
    embs = model.encode(words, convert_to_numpy=True)
    sim = cosine_similarity(embs)
    n = len(words)
    total, count = 0.0, 0
    for i in range(n):
        for j in range(i+1, n):
            total += sim[i][j]
            count += 1
    return total / count if count else 0.0

In [ ]:
# Compute similarity for every group in the dataset
from tqdm import tqdm

COLOR_ORDER = ['yellow', 'green', 'blue', 'purple']
COLOR_HEX = {'yellow': '#F9DF6D', 'green': '#A0C35A', 'blue': '#B0C4EF', 'purple': '#BA81C5'}

group_rows = []
for pi, p in enumerate(tqdm(raw, desc='Computing similarities')):
    puzzle_groups = []
    for ai, ans in enumerate(p['answers']):
        sim = avg_pairwise_sim(ans['words'], model)
        puzzle_groups.append({
            'puzzle_idx': pi,
            'date': p['date'],
            'answer_idx': ai,  # 0=yellow, 1=green, 2=blue, 3=purple (NYT ordering)
            'category': ans['answerDescription'],
            'words': ans['words'],
            'nyt_color': COLOR_ORDER[ai],
            'similarity': float(sim),
        })

    # Also assign colors by ranking (highest sim = yellow)
    ranked = sorted(puzzle_groups, key=lambda g: g['similarity'], reverse=True)
    for rank, g in enumerate(ranked):
        g['ranked_color'] = COLOR_ORDER[rank]

    group_rows.extend(puzzle_groups)

df_groups = pd.DataFrame(group_rows)
print(f'\nComputed similarity for {len(df_groups)} groups across {len(raw)} puzzles')

In [ ]:
# Compare NYT ordering vs similarity ranking
print('Similarity statistics by NYT color (answer position 0–3):')
print('=' * 65)
print(f'{"Color":>8}  {"Mean":>7}  {"Median":>7}  {"Std":>7}  {"Count":>5}')
print('-' * 65)

nyt_color_stats = {}
for color in COLOR_ORDER:
    subset = df_groups[df_groups['nyt_color'] == color]['similarity']
    nyt_color_stats[color] = {
        'mean': round(float(subset.mean()), 4),
        'median': round(float(subset.median()), 4),
        'std': round(float(subset.std()), 4),
        'count': len(subset),
    }
    print(f'{color:>8}  {subset.mean():7.4f}  {subset.median():7.4f}  {subset.std():7.4f}  {len(subset):5d}')

print('\nSimilarity statistics by RANKED color (highest → yellow):')
print('=' * 65)
print(f'{"Color":>8}  {"Mean":>7}  {"Median":>7}  {"Std":>7}  {"Count":>5}')
print('-' * 65)

ranked_color_stats = {}
for color in COLOR_ORDER:
    subset = df_groups[df_groups['ranked_color'] == color]['similarity']
    ranked_color_stats[color] = {
        'mean': round(float(subset.mean()), 4),
        'median': round(float(subset.median()), 4),
        'std': round(float(subset.std()), 4),
        'count': len(subset),
    }
    print(f'{color:>8}  {subset.mean():7.4f}  {subset.median():7.4f}  {subset.std():7.4f}  {len(subset):5d}')

STATS['embeddings'] = {
    'by_nyt_color': nyt_color_stats,
    'by_ranked_color': ranked_color_stats,
}

In [ ]:
# Plot: similarity distributions by NYT color
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# Paper reference thresholds
paper_means = {'yellow': 0.40, 'green': 0.35, 'blue': 0.29, 'purple': 0.27}

# LEFT: by NYT position
for color in COLOR_ORDER:
    subset = df_groups[df_groups['nyt_color'] == color]['similarity']
    axes[0].hist(subset, bins=30, alpha=0.45, label=color, color=COLOR_HEX[color], edgecolor='none')
    axes[0].axvline(subset.mean(), color=COLOR_HEX[color], ls='-', linewidth=2)

axes[0].set_xlabel('Avg Pairwise Cosine Similarity')
axes[0].set_ylabel('Count')
axes[0].set_title('Similarity by NYT Color (answer position)')
axes[0].legend()

# RIGHT: by ranked color
for color in COLOR_ORDER:
    subset = df_groups[df_groups['ranked_color'] == color]['similarity']
    axes[1].hist(subset, bins=30, alpha=0.45, label=color, color=COLOR_HEX[color], edgecolor='none')
    # Paper reference line
    axes[1].axvline(paper_means[color], color=COLOR_HEX[color], ls='--', linewidth=1.5)
    axes[1].axvline(subset.mean(), color=COLOR_HEX[color], ls='-', linewidth=2)

axes[1].set_xlabel('Avg Pairwise Cosine Similarity')
axes[1].set_ylabel('Count')
axes[1].set_title('Similarity by Ranked Color (highest sim → yellow)\nDashed = paper thresholds')
axes[1].legend()

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Box plots comparing NYT color vs ranked color
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col_field, title in [
    (axes[0], 'nyt_color', 'By NYT Position'),
    (axes[1], 'ranked_color', 'By Similarity Rank'),
]:
    box_data = [df_groups[df_groups[col_field] == c]['similarity'].values for c in COLOR_ORDER]
    bp = ax.boxplot(box_data, labels=[c.capitalize() for c in COLOR_ORDER],
                    patch_artist=True, widths=0.6)
    for patch, color in zip(bp['boxes'], COLOR_ORDER):
        patch.set_facecolor(COLOR_HEX[color])
        patch.set_alpha(0.7)
    # Paper reference markers
    for i, color in enumerate(COLOR_ORDER):
        ax.plot(i+1, paper_means[color], 'D', color='black', markersize=7, zorder=5)
    ax.set_ylabel('Avg Pairwise Cosine Similarity')
    ax.set_title(title)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('Cosine Similarity by Difficulty Color (♦ = paper reference values)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# How often does NYT ordering match similarity ranking?
match_count = 0
for _, g in df_groups.iterrows():
    if g['nyt_color'] == g['ranked_color']:
        match_count += 1

print(f'NYT color matches similarity rank: {match_count}/{len(df_groups)} ({match_count/len(df_groups):.1%})')
print('(i.e., how often the NYT\'s easiest-to-hardest ordering aligns with embedding similarity)')

STATS['embeddings']['nyt_rank_agreement'] = round(match_count / len(df_groups), 4)

In [ ]:
# Overall similarity distribution with paper thresholds
fig, ax = plt.subplots(figsize=(12, 5))

ax.hist(df_groups['similarity'], bins=50, color='steelblue', edgecolor='white', alpha=0.7)

for color, val in paper_means.items():
    ax.axvline(val, color=COLOR_HEX[color], ls='--', linewidth=2,
               label=f'{color} threshold ({val})')

ax.set_xlabel('Avg Pairwise Cosine Similarity')
ax.set_ylabel('Number of Groups')
ax.set_title('Overall Group Similarity Distribution (2,216 groups)')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

STATS['embeddings']['overall_mean'] = round(float(df_groups['similarity'].mean()), 4)
STATS['embeddings']['overall_std'] = round(float(df_groups['similarity'].std()), 4)

---
## 6. Cross-Puzzle Patterns

In [ ]:
# Most "deceptive" words: appear in many different category types
word_to_cats = defaultdict(set)  # word -> set of categories it appeared in
word_to_cat_types = defaultdict(set)  # word -> set of category types

for i, p in enumerate(raw):
    for ai, ans in enumerate(p['answers']):
        cat = ans['answerDescription'].upper()
        ctype = classify_category(cat)
        for w in ans['words']:
            wu = w.upper()
            word_to_cats[wu].add(cat)
            word_to_cat_types[wu].add(ctype)

# Words that appear in the most distinct categories
versatile = [(w, len(cats), cats) for w, cats in word_to_cats.items() if len(cats) >= 3]
versatile.sort(key=lambda x: -x[1])

print('Most "deceptive" words (appear in 3+ distinct categories):')
print(f'Total: {len(versatile)} words\n')
for w, n, cats in versatile[:20]:
    cats_short = list(cats)[:4]
    more = f' (+{n-4} more)' if n > 4 else ''
    print(f'  {w:20s}  {n} categories: {", ".join(cats_short)}{more}')

STATS['cross_puzzle'] = {
    'versatile_words_3plus_cats': len(versatile),
    'top_10_versatile': [{'word': w, 'num_categories': n} for w, n, _ in versatile[:10]],
}

In [ ]:
# Word co-occurrence: which words appear together most often across puzzles?
from itertools import combinations

pair_counts = Counter()
for p in raw:
    for ans in p['answers']:
        words_upper = sorted(w.upper() for w in ans['words'])
        for pair in combinations(words_upper, 2):
            pair_counts[pair] += 1

# Pairs appearing 2+ times
repeat_pairs = [(pair, c) for pair, c in pair_counts.most_common() if c >= 2]
print(f'Word pairs in the same group 2+ times: {len(repeat_pairs)}')
print('\nTop 20 most co-occurring word pairs (same group):')
for (w1, w2), c in repeat_pairs[:20]:
    print(f'  {w1:15s} + {w2:15s}  {c}×')

STATS['cross_puzzle']['repeat_pairs'] = len(repeat_pairs)

In [ ]:
# Category co-occurrence within puzzles: which category types tend to appear together?
puzzle_type_combos = []
for p in raw:
    types = sorted(classify_category(a['answerDescription']) for a in p['answers'])
    puzzle_type_combos.append(tuple(types))

# Count pairs of category types within each puzzle
type_pair_counts = Counter()
for types in puzzle_type_combos:
    for pair in combinations(types, 2):
        type_pair_counts[pair] += 1

# Build a co-occurrence matrix
all_types = sorted(set(t for types in puzzle_type_combos for t in types))
cooc = pd.DataFrame(0, index=all_types, columns=all_types)
for (t1, t2), c in type_pair_counts.items():
    cooc.loc[t1, t2] += c
    cooc.loc[t2, t1] += c

# Self-occurrence: how many puzzles have 2+ groups of the same type?
for types in puzzle_type_combos:
    tc = Counter(types)
    for t, n in tc.items():
        if n >= 2:
            cooc.loc[t, t] += 1

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cooc, annot=True, fmt='d', cmap='YlOrRd', ax=ax,
            linewidths=0.5, linecolor='white', cbar_kws={'label': 'Co-occurrences'})
ax.set_title('Category Type Co-occurrence Within Puzzles')
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Category type distribution by difficulty color (NYT position)
color_type_data = []
for p in raw:
    for ai, ans in enumerate(p['answers']):
        color_type_data.append({
            'color': COLOR_ORDER[ai],
            'type': classify_category(ans['answerDescription']),
        })

df_ct = pd.DataFrame(color_type_data)
ct_pivot = df_ct.groupby(['color', 'type']).size().unstack(fill_value=0)
# Normalize per color
ct_pct = ct_pivot.div(ct_pivot.sum(axis=1), axis=0) * 100

# Reindex to color order
ct_pct = ct_pct.reindex(COLOR_ORDER)

fig, ax = plt.subplots(figsize=(12, 5))
ct_pct.plot(kind='bar', stacked=True, ax=ax, color=sns.color_palette('Set2', len(ct_pct.columns)),
            edgecolor='white', linewidth=0.5)
ax.set_ylabel('% of categories')
ax.set_xlabel('Difficulty Color (NYT position)')
ax.set_title('Category Type Distribution by Difficulty Color')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.set_xticklabels([c.capitalize() for c in COLOR_ORDER], rotation=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print('\nCategory type % by color:')
print(ct_pct.round(1).to_string())

In [ ]:
# Save all statistics
out_path = ROOT / 'data' / 'nyt_analysis_stats.json'
with open(out_path, 'w') as f:
    json.dump(STATS, f, indent=2, default=str)

print(f'Saved analysis statistics to {out_path}')
print(f'\nKeys: {list(STATS.keys())}')
print(json.dumps(STATS, indent=2, default=str)[:2000])

In [ ]:
# Final summary
print('=' * 65)
print('NYT CONNECTIONS DATASET — KEY FINDINGS')
print('=' * 65)
print(f'Puzzles analyzed:          {n_puzzles}')
print(f'Unique words:              {len(unique_words):,}')
print(f'Unique categories:         {unique_cats:,}')
print(f'Difficulty range:          {STATS["difficulty"]["min"]} – {STATS["difficulty"]["max"]}')
print(f'Difficulty trend:          r = {STATS["difficulty"]["time_correlation"]:.3f}')
print()
print('Embedding similarity by ranked color vs paper thresholds:')
for c in COLOR_ORDER:
    actual = STATS['embeddings']['by_ranked_color'][c]['mean']
    expected = paper_means[c]
    diff = actual - expected
    print(f'  {c:>6}: actual={actual:.4f}  expected={expected:.2f}  diff={diff:+.4f}')
print()
print(f'NYT color ↔ rank agreement: {STATS["embeddings"]["nyt_rank_agreement"]:.1%}')
print(f'Versatile words (3+ cats):  {STATS["cross_puzzle"]["versatile_words_3plus_cats"]}')
print(f'Repeat word pairs:          {STATS["cross_puzzle"]["repeat_pairs"]}')
print('=' * 65)